# 🔥 MapBiomas Fire — Sentinel-2 Monitor
## Piloto Perú · Google Colab

**Flujo de campaña semanal:**  
Semana 1 → Mosaico + Muestras + Entrenamiento  
Semana 2 → Clasificación + Publicación

| Módulo | Descripción |
|--------|-----------|
| M0 | Autenticación y Configuración |
| M1a | Despachador de Exportaciones (GEE → GCS) |
| M1b | Ensamblador de Mosaicos (GCS Shards → COG) |
| M2 | Gestor de Muestras |
| M3 | Entrenamiento del Modelo DNN |
| M4 | Clasificación por Campaña |
| M5 | Máscara LULC + Publicación |

---
> **Resultado:** Píxeles quemados = `dayOfYear` (día juliano del mes) · No quemados = `0`  
> **Máscara LULC:** clases 26, 22, 33, 24 (MapBiomas Perú Colección 2) — aplicada post-clasificación

## ⚙️ M0 — Instalación y Autenticación

In [ ]:
# ── 1. Clonar repositorio
!git clone https://github.com/mapbiomas/peru-fire.git /content/peru-fire

In [ ]:
# ── 2. Instalar dependencias (ejecutar una vez por sesión)
!pip install -q earthengine-api gcsfs rasterio scipy
!pip install -q tensorflow==2.13.0  # TF2 con compatibilidad TF1 vía compat.v1
!apt-get -qq install -y gdal-bin python3-gdal > /dev/null

In [ ]:
import sys, os

# ── Configurar rutas del repositorio clonado
REPO_PATH = '/content/peru-fire' 
SRC_PATH  = os.path.join(REPO_PATH, 'mapbiomas_fire_monitor', 'version_01', 'src', 'colab')

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(f'✅ sys.path configurado: {SRC_PATH}')

In [ ]:
# ── Autenticación GEE + GCS
from google.colab import auth as colab_auth
colab_auth.authenticate_user()  # autentica ADC para GCS automáticamente

from M0_auth_config import CONFIG, authenticate, print_config
authenticate()   # inicializa ee con project
print_config()

## 🛰️ M1 — Generador y Ensamblaje de Mosaicos

### 🛰️ M1a — Despachador de Exportaciones Inteligente (GEE → GCS)

In [ ]:
# ── Interfaz para enviar mosaicos faltantes o años completos
from M1a_export_dispatcher import run_ui as run_exporter
run_exporter()

### 🗺️ M1b — Ensamblador Nacional Inteligente (GCS Shards → COG)

In [ ]:
# ── Interfaz para ensamblar fragmentos y ver panel de descargas
from M1b_mosaic_assembler import run_ui as run_assembler
run_assembler()

In [ ]:
# ── (Opcional) Monitorear tareas de GEE enviadas
import ee
tasks = ee.batch.Task.list()
for t in tasks[:10]:
    s = t.status()
    print(f"  {s['state']:12s} | {s['description']}")

## 🧬 M2 — Gestor de Muestras

In [ ]:
# ── Selección de muestras vectoriales + bandas del modelo
from M2_sample_manager import run_ui as sample_ui
sample_interface = sample_ui()

In [ ]:
# ── Recuperar la selección confirmada de M2
sample_fc, selected_bands = sample_interface.get_selection()
print(f'✅ {sample_fc.size().getInfo():,} muestras | Bandas: {selected_bands}')

## 🧠 M3 — Entrenamiento del Modelo DNN

In [ ]:
# ── Entrenamiento de la DNN con las bandas seleccionadas en M2
from M3_model_trainer import run_ui as trainer_ui
trainer_interface = trainer_ui(
    sample_fc=sample_fc,
    selected_bands=selected_bands
)

## 🔥 M4 — Clasificación por Campaña

In [ ]:
# ── Clasificación por cuadrícula dinámica y por región
from M4_classifier import run_ui as classifier_ui
classifier_interface = classifier_ui()

## 📢 M5 — Máscara LULC y Publicación

In [ ]:
# ── Post-clasificación → publicación en GEE
from M5_publisher import run_ui as publisher_ui
publisher_interface = publisher_ui()

---
## 📋 Flujo Operativo de la Campaña

```
SEMANA 1 (primeros ~5 días hábiles)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Día 1 │ M0: git clone + instalar + autenticar
Día 1 │ M1a: verificar estado → enviar mosaicos faltantes (tareas de GEE)
Día 2 │ M1b: verificar shards en GCS → ensamblar mosaico del país (VRT→COG)
Día 2 │ M2: seleccionar muestras + confirmar bandas del modelo
Día 3 │ M3: entrenar modelo (saltar si el modelo existente está bien)
Día 3 │ M4: enviar clasificación por región + cuadrícula multianual

SEMANA 2 (hasta el día 10)
━━━━━━━━━━━━━━━━━━━━
Día 6 │ M4: verificar fragmentos clasificados
Día 6 │ M5: construir mosaico de clasificación del país
Día 7 │ M5: revisión → borrador o final → publicar en GEE
```
